# CNV Preprocessing — METABRIC
Input: `preprocessed/metabric_cnv_cleaned.csv`

Output: `03_METABRIC_external_validation/cnv/statistical_filtered/cnv_8_composite_Ngenes.csv`

In [2]:
from pathlib import Path

_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "preprocessed" / "metabric_cnv_cleaned.csv").exists()
)

CNV_FILE     = BASE_DIR / "preprocessed" / "metabric_cnv_cleaned.csv"
OUTCOME_FILE = BASE_DIR / "preprocessed" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "cnv" / "statistical_filtered"
STATS_CACHE  = BASE_DIR / "cnv" / "cnv_statistics_cache.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_BROAD = 1000
MIN_GENES    = 30

assert CNV_FILE.exists(),     f"Not found: {CNV_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base   : {BASE_DIR}")
print(f"Output : {OUTPUT_DIR}")
print("Paths OK")

Base   : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation
Output : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\cnv\statistical_filtered
Paths OK


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import mutual_info_regression
import warnings
warnings.filterwarnings("ignore")
print("Imports OK")

Imports OK


## 1. Load data

In [5]:
print("Loading CNV data...")
cnv = pd.read_csv(CNV_FILE, index_col=0)
print(f"  Shape  : {cnv.shape}  (samples x genes)")

print("\nLoading outcome...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Samples : {len(outcome)}")
print(f"  Events  : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

common  = sorted(set(cnv.index) & set(outcome.index))
cnv     = cnv.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f"  Overlap : {len(common)} samples")

Loading CNV data...
  Shape  : (1980, 22542)  (samples x genes)

Loading outcome...
  Samples : 1980
  Events  : 1143 (57.7%)
  Overlap : 1980 samples


## 2. Quality control & variance pre-filter

In [7]:
# Missing values
miss_frac = cnv.isna().mean(axis=0)
cnv = cnv.loc[:, miss_frac <= 0.20]
print(f"After missing filter (>20%) : {cnv.shape[1]:,} genes")

# Impute residual missing with median
cnv = cnv.fillna(cnv.median(axis=0))

# Remove zero-variance
var = cnv.var(axis=0)
cnv = cnv.loc[:, var > 0]
print(f"After zero-variance removal : {cnv.shape[1]:,} genes")

# Remove bottom 25% variance
variances     = cnv.var(axis=0)
var_threshold = variances.quantile(0.25)
cnv_var       = cnv.loc[:, variances > var_threshold]
print(f"After bottom-25% removal   : {cnv_var.shape[1]:,} genes  (threshold={var_threshold:.4f})")

print(f"\nValue range: {cnv_var.values.min():.2f} – {cnv_var.values.max():.2f}")

After missing filter (>20%) : 22,542 genes
After zero-variance removal : 22,542 genes
After bottom-25% removal   : 16,906 genes  (threshold=0.2174)

Value range: -2.00 – 2.00


## 3. Statistical scoring — Spearman + MI + Distance correlation

In [9]:
if STATS_CACHE.exists():
    print(f"Loading cached stats from {STATS_CACHE.name}...")
    stats = pd.read_csv(STATS_CACHE)
    stats = stats[stats["gene"].isin(cnv_var.columns)].reset_index(drop=True)
    print(f"  Loaded {len(stats):,} genes")

else:
    genes   = cnv_var.columns.tolist()
    os_time = outcome["OS.time"].values
    n       = len(genes)
    print(f"Computing stats for {n:,} genes...")

    print("  [1/3] Spearman...")
    rho_arr  = np.zeros(n)
    pval_arr = np.zeros(n)
    for i, g in enumerate(genes):
        r, p        = spearmanr(cnv_var[g].values, os_time)
        rho_arr[i]  = r if not np.isnan(r) else 0.0
        pval_arr[i] = p if not np.isnan(p) else 1.0
        if (i + 1) % 3000 == 0:
            print(f"    {i+1:,} / {n:,}")
    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")
    print(f"    FDR < 0.05: {(fdr_arr < 0.05).sum()} genes")

    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(cnv_var.values, os_time, random_state=42, n_neighbors=5)

    print("  [3/3] Distance correlation (rank approximation)...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = [
        abs(float(np.corrcoef(rankdata(cnv_var[g].values).astype(float), os_rank)[0, 1]))
        for g in genes
    ]

    stats = pd.DataFrame({
        "gene":          genes,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "variance":      variances[genes].values,
    })

    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]) / 3.0
    stats = stats.sort_values("composite").reset_index(drop=True)

    STATS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE}")

print(f"\nFDR < 0.05 : {int((stats['fdr'] < 0.05).sum()):,} / {len(stats):,} genes")
print(f"|rho| range: {stats['abs_spearman'].min():.4f} – {stats['abs_spearman'].max():.4f}")
print(f"\nTop 10 by composite score:")
print(stats[["gene", "abs_spearman", "mutual_info", "distance_corr", "composite"]].head(10).to_string(index=False))

Computing stats for 16,906 genes...
  [1/3] Spearman...
    3,000 / 16,906
    6,000 / 16,906
    9,000 / 16,906
    12,000 / 16,906
    15,000 / 16,906
    FDR < 0.05: 6908 genes
  [2/3] Mutual information...
  [3/3] Distance correlation (rank approximation)...
  Cached to C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\cnv\cnv_statistics_cache.csv

FDR < 0.05 : 6,908 / 16,906 genes
|rho| range: 0.0000 – 0.1460

Top 10 by composite score:
  gene  abs_spearman  mutual_info  distance_corr  composite
 PSMB6      0.120742     0.019985       0.120742 351.000000
  PLD2      0.120742     0.019884       0.120742 358.666667
  CCL7      0.112062     0.028528       0.112062 368.166667
 CCL11      0.112062     0.028528       0.112062 368.166667
  CCL2      0.112062     0.028140       0.112062 373.666667
 SPAG7      0.120274     0.019652       0.120274 386.666667
SCNN1B      0.114211     0.023655       0.114211 388.500000
  COG7      0.114211     0.023655       0.114211 388.500000

## 4. Build V8 composite dataset

In [11]:
top_genes = stats.head(TARGET_BROAD)["gene"].tolist()
top_genes = [g for g in top_genes if g in cnv_var.columns]

if len(top_genes) < MIN_GENES:
    have = set(top_genes)
    top_genes += [g for g in stats["gene"] if g not in have][:MIN_GENES - len(top_genes)]
    print(f"Padded to {len(top_genes)} genes")

df_v8 = cnv_var[top_genes]

fname = f"cnv_8_composite_{len(df_v8.columns)}genes.csv"
df_v8.to_csv(OUTPUT_DIR / fname)
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "cnv_statistics_complete.csv", index=False)

print(f"Genes   : {len(df_v8.columns)}")
print(f"Samples : {len(df_v8)}")
print(f"Saved   : {fname}")

Genes   : 1000
Samples : 1980
Saved   : cnv_8_composite_1000genes.csv


## 5. Verification

In [13]:
df_check = pd.read_csv(OUTPUT_DIR / fname, index_col=0)

assert list(df_check.index) == list(outcome.index), "Index mismatch!"
assert df_check.isna().sum().sum() == 0, "NaN values found!"
assert np.isinf(df_check.values).sum() == 0, "Inf values found!"

print(f"Shape  : {df_check.shape}")
print(f"No NaN : OK")
print(f"No Inf : OK")
print(f"Index  : OK")
print(f"\nNext step: run 03_run_MB_cnv_METABRIC.py")

Shape  : (1980, 1000)
No NaN : OK
No Inf : OK
Index  : OK

Next step: run 03_run_MB_cnv_METABRIC.py
